- Đọc data
- Xử lý clean data, oulier
- Chia train, test
- Scale data
- Train

In [ ]:
import pandas as pd

file_path = '/content/drive/MyDrive/DATN/fetal_health.csv'
data = pd.read_csv(file_path)
display(data.head())

,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency,fetal_health
0,120.0,0.000,0.0,0.000,0.000,0.0,0.0,73.0,0.5,43.0,...,62.0,126.0,2.0,0.0,120.0,137.0,121.0,73.0,1.0,2.0
1,132.0,0.006,0.0,0.006,0.003,0.0,0.0,17.0,2.1,0.0,...,68.0,198.0,6.0,1.0,141.0,136.0,140.0,12.0,0.0,1.0
2,133.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.1,0.0,...,68.0,198.0,5.0,1.0,141.0,135.0,138.0,13.0,0.0,1.0
3,134.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,11.0,0.0,137.0,134.0,137.0,13.0,1.0,1.0
4,132.0,0.007,0.0,0.008,0.000,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,9.0,0.0,137.0,136.0,138.0,11.0,1.0,1.0


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2126 entries, 0 to 2125
Data columns (total 22 columns):
 #   Column                                                  Non-Null Count  Dtype  
---  ------                                                  --------------  -----  
 0   baseline value                                          2126 non-null   float64
 1   accelerations                                           2126 non-null   float64
 2   fetal_movement                                          2126 non-null   float64
 3   uterine_contractions                                    2126 non-null   float64
 4   light_decelerations                                     2126 non-null   float64
 5   severe_decelerations                                    2126 non-null   float64
 6   prolongued_decelerations                                2126 non-null   float64
 7   abnormal_short_term_variability                         2126 non-null   float64
 8   mean_value_of_short_term_variability  

In [ ]:
data.shape

(2126, 22)

In [ ]:
data.duplicated().sum()

np.int64(13)

In [ ]:
data.drop_duplicates(inplace=True)

In [ ]:
data.duplicated().sum()

np.int64(0)

In [ ]:
data.isna().sum().sum()

np.int64(0)

In [ ]:
data.describe().T

,count,mean,std,min,25%,50%,75%,max
baseline value,2113.0,133.304780,9.837451,106.0,126.000,133.000,140.000,160.000
accelerations,2113.0,0.003188,0.003871,0.0,0.000,0.002,0.006,0.019
fetal_movement,2113.0,0.009517,0.046804,0.0,0.000,0.000,0.003,0.481
uterine_contractions,2113.0,0.004387,0.002941,0.0,0.002,0.005,0.007,0.015
light_decelerations,2113.0,0.001901,0.002966,0.0,0.000,0.000,0.003,0.015
severe_decelerations,2113.0,0.000003,0.000057,0.0,0.000,0.000,0.000,0.001
prolongued_decelerations,2113.0,0.000159,0.000592,0.0,0.000,0.000,0.000,0.005
abnormal_short_term_variability,2113.0,46.993848,17.177782,12.0,32.000,49.000,61.000,87.000
mean_value_of_short_term_variability,2113.0,1.335021,0.884368,0.2,0.700,1.200,1.700,7.000
percentage_of_time_with_abnormal_long_term_variability,2113.0,9.795078,18.337073,0.0,0.000,0.000,11.000,91.000


In [ ]:
negative_values_found = False
for column in data.select_dtypes(include=['number']).columns:
    negative_count = (data[column] < 0).sum()
    if negative_count > 0:
        print(f"Cột '{column}' có {negative_count} giá trị nhỏ hơn 0.")
        negative_values_found = True

if not negative_values_found:
    print("Không tìm thấy giá trị nào nhỏ hơn 0 trong các cột số.")

Cột 'histogram_tendency' có 165 giá trị nhỏ hơn 0.


In [ ]:
data["histogram_tendency"].value_counts()

,count
histogram_tendency,
0.0,1110
1.0,838
-1.0,165


In [ ]:
data["fetal_health"].value_counts()

,count
fetal_health,
1.0,1646
2.0,292
3.0,175


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import zscore
from sklearn.model_selection import train_test_split

# Tách X, y
X = data.drop('fetal_health', axis=1)
y = data['fetal_health']

# Chỉ lấy các cột số
num_cols = X.select_dtypes(include=['int64', 'float64']).columns

# Tính Z-score trên toàn bộ X
z_scores = np.abs(zscore(X[num_cols], nan_policy='omit'))

# Chuyển thành DataFrame để giữ index
z_scores = pd.DataFrame(z_scores, columns=num_cols, index=X.index)

# Ngưỡng lọc outlier
threshold = 3.5

# Giữ lại các dòng không chứa outlier
mask = (z_scores < threshold).all(axis=1)

X_clean = X.loc[mask].copy()
y_clean = y.loc[mask].copy()

print(f"Kích thước X trước khi lọc Z-score: {X.shape}")
print(f"Kích thước y trước khi lọc Z-score: {y.shape}")
print(f"Kích thước X sau khi lọc Z-score: {X_clean.shape}")
print(f"Kích thước y sau khi lọc Z-score: {y_clean.shape}")

# Chia train/test sau khi đã lọc
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y_clean,
    test_size=0.2,
    random_state=42,
    stratify=y_clean
)

print(f"Kích thước tập huấn luyện X: {X_train.shape}")
print(f"Kích thước tập kiểm tra X: {X_test.shape}")
print(f"Kích thước tập huấn luyện y: {y_train.shape}")
print(f"Kích thước tập kiểm tra y: {y_test.shape}")

Kích thước X trước khi lọc Z-score: (2113, 21)
Kích thước y trước khi lọc Z-score: (2113,)
Kích thước X sau khi lọc Z-score: (1920, 21)
Kích thước y sau khi lọc Z-score: (1920,)
Kích thước tập huấn luyện X: (1536, 21)
Kích thước tập kiểm tra X: (384, 21)
Kích thước tập huấn luyện y: (1536,)
Kích thước tập kiểm tra y: (384,)


In [ ]:
# from sklearn.preprocessing import StandardScaler

# # Khởi tạo scaler
# scaler = StandardScaler()

# # Fit trên tập train, transform cả train và test
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

# # Nếu muốn giữ dạng DataFrame để dễ nhìn tên cột
# X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
# X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

# print("Đã scale dữ liệu xong")
# display(X_train_scaled.head())

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Khởi tạo scaler
scaler = MinMaxScaler()

# Fit trên train, transform cả train và test
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Nếu muốn giữ DataFrame
import pandas as pd
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
print("Đã scale dữ liệu xong")
display(X_train_scaled.head())

Đã scale dữ liệu xong


,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_width,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency
573,0.407407,0.5625,0.000000,0.500000,0.166667,0.0,0.0,0.333333,0.928571,0.000000,...,0.903226,0.018349,0.846154,0.500000,0.5,0.47,0.656863,0.505376,0.840336,0.5
1502,0.444444,0.2500,0.000000,0.571429,0.083333,0.0,0.0,0.373333,0.166667,0.232877,...,0.245161,0.550459,0.362637,0.428571,0.0,0.50,0.519608,0.451613,0.033613,0.5
1279,0.166667,0.8125,0.000000,0.142857,0.000000,0.0,0.0,0.093333,0.380952,0.000000,...,0.316129,0.467890,0.384615,0.071429,0.5,0.34,0.401961,0.301075,0.075630,0.5
1858,0.592593,1.0000,0.000000,0.357143,0.000000,0.0,0.0,0.520000,0.166667,0.000000,...,0.270968,0.660550,0.538462,0.214286,0.0,0.61,0.676471,0.612903,0.050420,0.5
755,0.444444,0.0000,0.014388,0.000000,0.000000,0.0,0.0,0.626667,0.047619,0.506849,...,0.090323,0.688073,0.263736,0.142857,0.0,0.43,0.509804,0.419355,0.016807,0.0


Random forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score

# Khởi tạo model
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

# Train model với dữ liệu đã scale
rf_model.fit(X_train_scaled, y_train)

# Predict
y_pred = rf_model.predict(X_test_scaled)

# Đánh giá
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision weighted:", precision_score(y_test, y_pred, average='weighted'))
print("Recall weighted:", recall_score(y_test, y_pred, average='weighted'))
print("F1 weighted:", f1_score(y_test, y_pred, average='weighted'))

print("Precision macro:", precision_score(y_test, y_pred, average='macro'))
print("Recall macro:", recall_score(y_test, y_pred, average='macro'))
print("F1 macro:", f1_score(y_test, y_pred, average='macro'))
print("\nDetail about one class:")
print("Classification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.953125
Precision weighted: 0.952054874296765
Recall weighted: 0.953125
F1 weighted: 0.9521193456769425
Precision macro: 0.8949507735583685
Recall macro: 0.8850684968521171
F1 macro: 0.8888659613585136

Detail about one class:
Classification Report:
               precision    recall  f1-score   support

         1.0       0.97      0.99      0.98       311
         2.0       0.88      0.79      0.83        56
         3.0       0.83      0.88      0.86        17

    accuracy                           0.95       384
   macro avg       0.89      0.89      0.89       384
weighted avg       0.95      0.95      0.95       384


Confusion Matrix:
 [[307   4   0]
 [  9  44   3]
 [  0   2  15]]


XGBoot

In [ ]:
y_train = y_train.astype(int) - 1
y_test = y_test.astype(int) - 1

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score

# Khởi tạo model XGBoost
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softmax',   # dùng cho phân loại nhiều lớp
    num_class=len(y_train.unique()),
    random_state=42,
    n_jobs=-1,
    eval_metric='mlogloss'
)

# Train model
xgb_model.fit(X_train_scaled, y_train)

# Predict
y_pred_xg = xgb_model.predict(X_test_scaled)

# Đánh giá
print("Accuracy:", accuracy_score(y_test, y_pred_xg))
print("Precision weighted:", precision_score(y_test, y_pred_xg, average='weighted'))
print("Recall weighted:", recall_score(y_test, y_pred_xg, average='weighted'))
print("F1 weighted:", f1_score(y_test, y_pred_xg, average='weighted'))

print("Precision macro:", precision_score(y_test, y_pred_xg, average='macro'))
print("Recall macro:", recall_score(y_test, y_pred_xg, average='macro'))
print("F1 macro:", f1_score(y_test, y_pred_xg, average='macro'))

print("\nDetail about one class:")
print("Classification Report:\n", classification_report(y_test, y_pred_xg))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_xg))

Accuracy: 0.9609375
Precision weighted: 0.960885830026455
Recall weighted: 0.9609375
F1 weighted: 0.9600519798952307
Precision macro: 0.91432350718065
Recall macro: 0.9302365640790078
F1 macro: 0.9189641489960979

Detail about one class:
Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.99      0.98       311
           1       0.92      0.80      0.86        56
           2       0.85      1.00      0.92        17

    accuracy                           0.96       384
   macro avg       0.91      0.93      0.92       384
weighted avg       0.96      0.96      0.96       384


Confusion Matrix:
 [[307   4   0]
 [  8  45   3]
 [  0   0  17]]
